# Example queries: `timeseries` (comstock_oedi_agg)

Auto-generated from `tests/query_snapshots/timeseries.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/comstock_oedi_agg_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/comstock_oedi_agg_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "comstock_amy2018_r2_2025",
    buildstock_type="comstock",
    db_schema="comstock_oedi_agg_state_and_county",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading comstock_amy2018_r2_2025 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `ts_monthly_electricity_by_state`

Monthly electricity grouped by state (CO only), upgrade 0 baseline.


In [3]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='month',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01,25743,84415.71204,2976,1.385482e+09
1,CO,2018-02-01,25743,84415.71204,2688,1.280966e+09
2,CO,2018-03-01,25743,84415.71204,2976,1.237644e+09
3,CO,2018-04-01,25743,84415.71204,2880,1.171385e+09
4,CO,2018-05-01,25743,84415.71204,2976,1.289014e+09


## `ts_monthly_two_fuels_by_building_type`

Monthly electricity + natural gas grouped by building type and time, CO only.


In [4]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption', 'out.natural_gas.total.energy_consumption'],
    annual_only=False,
    timestamp_grouping_func='month',
    group_by=['comstock_building_type', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption,natural_gas.total.energy_consumption
0,FullServiceRestaurant,2018-01-01,2526,4556.431396,2976,1.042121e+08,1.283346e+08
1,FullServiceRestaurant,2018-02-01,2526,4556.431396,2688,9.695190e+07,1.206407e+08
2,FullServiceRestaurant,2018-03-01,2526,4556.431396,2976,9.693666e+07,1.067041e+08
3,FullServiceRestaurant,2018-04-01,2526,4556.431396,2880,9.240466e+07,9.565136e+07
4,FullServiceRestaurant,2018-05-01,2526,4556.431396,2976,9.842727e+07,8.398370e+07


## `ts_year_collapse_electricity`

Year-collapsed timeseries electricity, grouped by building type, CO only.


In [5]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    timestamp_grouping_func='year',
    group_by=['comstock_building_type'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,FullServiceRestaurant,2526,4556.431396,35040,1.209201e+09
1,Hospital,63,152.003100,35040,1.369473e+09
2,LargeHotel,1460,3317.336942,35040,1.128185e+09
3,LargeOffice,866,849.927533,35040,1.546956e+09
4,MediumOffice,2351,2418.404703,35040,1.298819e+09


## `ts_15min_raw_electricity_by_state`

Raw 15-min cadence timeseries (no timestamp_grouping_func), CO only. Exercises the un-aggregated time-axis code path that example notebooks rely on.


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,electricity.total.energy_consumption
0,CO,2018-01-01 00:15:00,25743,84415.71204,558329.416935
1,CO,2018-01-01 00:30:00,25743,84415.71204,556124.833268
2,CO,2018-01-01 00:45:00,25743,84415.71204,552236.206143
3,CO,2018-01-01 01:00:00,25743,84415.71204,550257.671238
4,CO,2018-01-01 01:15:00,25743,84415.71204,542227.004333


## `ts_daily_electricity_by_state`

Daily-grouped timeseries electricity, CO only. Exercises date_trunc('day', ...) plus the rows_per_sample / distinct-bs-keys grouping_metrics branch.


In [7]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='day',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01,25743,84415.71204,96,5.631904e+07
1,CO,2018-01-02,25743,84415.71204,96,5.220220e+07
2,CO,2018-01-03,25743,84415.71204,96,4.810277e+07
3,CO,2018-01-04,25743,84415.71204,96,4.638417e+07
4,CO,2018-01-05,25743,84415.71204,96,4.562267e+07


## `ts_hourly_electricity_by_building_type`

Hourly-grouped timeseries electricity, CO + single building type. Exercises date_trunc('hour', ...) — the smallest grouping_func granularity.


In [8]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='hour',
    group_by=['comstock_building_type', 'time'],
    restrict=[('state', ['CO']), ('comstock_building_type', ['Warehouse'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,Warehouse,2018-01-01 00:00:00,2494,21505.528073,4,425794.631062
1,Warehouse,2018-01-01 01:00:00,2494,21505.528073,4,430400.931908
2,Warehouse,2018-01-01 02:00:00,2494,21505.528073,4,364028.417298
3,Warehouse,2018-01-01 03:00:00,2494,21505.528073,4,371733.766486
4,Warehouse,2018-01-01 04:00:00,2494,21505.528073,4,375253.338486


## `ts_monthly_multi_attribute_group_by`

Monthly TS electricity grouped by [building_type, state, time] — multi-attribute group_by combined with the time axis. Every other TS entry uses single-attribute + time; this pins the SQL shape for the more general case.


In [9]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='month',
    group_by=['comstock_building_type', 'state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,comstock_building_type,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,FullServiceRestaurant,CO,2018-01-01,2526,4556.431396,2976,1.042121e+08
1,FullServiceRestaurant,CO,2018-02-01,2526,4556.431396,2688,9.695190e+07
2,FullServiceRestaurant,CO,2018-03-01,2526,4556.431396,2976,9.693666e+07
3,FullServiceRestaurant,CO,2018-04-01,2526,4556.431396,2880,9.240466e+07
4,FullServiceRestaurant,CO,2018-05-01,2526,4556.431396,2976,9.842727e+07


## `ts_hourly_electricity_by_state`

Hourly TS electricity grouped by state, CO only. Restrict and group_by deliberately match ts_daily_electricity_by_state and ts_monthly_electricity_by_state so the hourly→daily→monthly sum invariants align.


In [10]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    annual_only=False,
    upgrade_id=0,
    timestamp_grouping_func='hour',
    group_by=['state', 'time'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,state,timestamp,sample_count,units_count,rows_per_sample,electricity.total.energy_consumption
0,CO,2018-01-01 00:00:00,25743,84415.71204,4,2.216948e+06
1,CO,2018-01-01 01:00:00,25743,84415.71204,4,2.177930e+06
2,CO,2018-01-01 02:00:00,25743,84415.71204,4,1.874028e+06
3,CO,2018-01-01 03:00:00,25743,84415.71204,4,1.832707e+06
4,CO,2018-01-01 04:00:00,25743,84415.71204,4,1.820178e+06
